# GRAFT — Graph Retrieval-Augmented Fine-Tuning

Train a causal language model (distilgpt2) to perform graph-grounded QA.

**Setup:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Step 1: Clone the repository and install
!git clone https://github.com/pacman-cli/Machine-Learning.git
%cd /content/Machine-Learning/graph-retrival-augmented-finetuning
!pip install -q torch transformers datasets tqdm pyyaml
!pip install -e .

In [ ]:
# Step 2: Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
# Step 3: Smoke test
!python -m scripts.smoke_test

In [ ]:
# Step 4: Train the model
!python -m scripts.train --config config/training_config.yaml

In [ ]:
# Step 5: Evaluate
!python -m scripts.evaluate --model_path ./pt_graft_model_save/best

In [ ]:
# Step 6: Test inference on a custom query
from src.domain.entities import SceneGraph, SceneObject, SceneRelation, KBTriple
from src.application.inference import GraftInferencePipeline

pipeline = GraftInferencePipeline(
    model_path="./pt_graft_model_save/best",
    base_model_id="distilgpt2",
    max_new_tokens=64,
)

scene = SceneGraph(
    objects={
        "1": SceneObject("1", "car", ["red", "parked"]),
        "2": SceneObject("2", "tree", ["large"]),
    },
    relations=[SceneRelation("1", "under", "2")],
)

facts = [
    KBTriple("tree", "CanDrop", "branches"),
    KBTriple("falling_branches", "Damage", "vehicle"),
]

result = pipeline.predict(scene, "What risk does the car face?", known_facts=facts)
print(f"Query: {result.query}")
print(f"Answer: {result.answer}")

In [ ]:
# Step 7 (Optional): Save model to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r ./pt_graft_model_save "/content/drive/MyDrive/graft_model_save"
print("Model saved to Google Drive!")